# [AdaBoost Working](https://youtu.be/RT0t9a3Xnfw?list=PLKnIA16_Rmvbr7zKYQuBfsVkjoLcJgxHH)  
1. Assign equal weight to every row in the dataset. its usuall 1/n with n being number of rows (ofc)
2. train a decision tree with max_depth = 1 on a data. make a prediction on the data... and then add the prediction column to dataframe (for better understanding). Now we have input columns, label column, prediction column and finally weight column.
3. Calculate its alpha... how?:
      <br> alright so we know that model that have higher rate of error are not to be given higher values of alpha because stumps with higher values of alpha gets to take more part in prediction making at the end, BUT its not that simple... you see the stumps with almost 100% error rate are always bound to make mistake in predicting... which means the opposite of what they predict will always be right... so we can use this to our advantage as well.
   Taking all of this into consideration now we know that we have to assign higher values of alpha to those stumps which have either error rate closer to 0% or closer to 100%.
#####  Look at the grpah downside of whiteboard.. it illustrates how values of alphas are supposed to be w.r.t values of error. if error close to 100% meaning model is making mistake almost everytime... we give a large value to alpha but NEGATIVE, and if value of error is small around 0% we give large value to alpha and its postive. if the error is somewhere around middle we give the alpha medium value...
![ytss](assets/2_ada_alpha.png)  

Now we need a mathematical function that can return values like these... lower error rate => higher postive returned value... higher error-rate => higher negative returned value, medium error-rate => medium returned value.  
**Formula**:   
`alpha = (1/2)ln((1-error)/error) `  
Using this error we find the value of alpha for every stump.  

### How to calculate error-rate?  
This is where we use the weight that we initialized in the first steps...   
`error = sum of weights of samples which are miss-categorized in model's view` 
if there are two data points out of 5 that stump predicted wrong and as we already know weight forevery sample is 1/5 which is 0.2, so error for this particular stump will be 0.2 + 0.2 = 4 (sum of weights of wrong predicted samples. e.g: this sample's orinal label is 1  but model said its 0)  

#### Look at the following ss to see calculation of error and using that alpha's  
<br>

![ytss](assets/3_ada_alpha.png)   
<br>

##### Here is it again:  

![ytss](assets/3_ada_alpha_2.png)

### With the alpha being found and attached to first decision stump, Now is the time to find new weights.  
1. First thing to do now is adjust the weights of all samples (miss-classified and correctly-classified.)
We have to increase the weight of miss-classified samples and decrease the weight of correctly classified samples. The question is how much to increase and how much to decrease...
Answer:

![ytss](assets/4_ada_weight.png)  

Now we have got the new values of all the samples... we assign these values to all of the rows  
## Updated:   
![ytss](assets/5_ada_weights.png)  
Sum of the weights always should be 1, as it was in the intial weights... so to do that to updated sums we normalize them, normalization can be achienved by deviding every number in the column with their sum, and that is whats in the normalized column.  

### Now that we have got weights updated, the last andmost important step of first stage is here, called: **UpSampling**.  

![ytss](assets/6_ada_upsampling.png)  

First thing to do to do upsampling is define ranges from weights... as it can be seen in last column: take first rane from 0 to the weight of that row... then for the next sample, start from previous sample's range and add this sample's weight to it and make that the end of the range. keep doing it for all of the samples... the last range will always end at 1... 

Now after that we take n number of random numbers between 0 to 1, lets say n is 5 we can do that in python by
```python
import numpy as np
np.random.rand(5,1)
#output:
#array([[0.69558431],
       [0.63797923],
       [0.11222008],
       [0.84775857],
       #[0.15814425]]) ofcourse result will be different number every time.
```

now that we have random numbers... take each number and find out in which range each of these numbers fall, and in whichever range they fall into, we note the index of that sample, whats the use of it? lets go back a little. when we increased the weights of missclassified samples, that will be helpful here cuz now the range of those samples which were miss-classified would be larger then the others so more of the randomly generated numbers will tend to fall into those ranges... and ultimately those samples would be noted again and again...  if there are any ranges that got left out, like none of the randomly generated numbers fell into those we can leave them... if multiple random numbers fall into the same range say k times, we note the index of that sample k times. Now through these noted indexes of samples we forge a new dataset and give that as input to the next decision stump, notice that its normal for this dataset to have same samples multiple times.  
And then all of the process is repeated until we reach the end of our decision stumps.  

### Prediction: Then finally we use the formula:   
**`h(x) = sign(alpha1*h1(x) + alpha2*h2(x) + alpha3*h3(x) + ... +alphaN*hN(x))`**   
if h(x) is positive it means the new query point or sample will be classified as +1, and if its negative it will classified as -1.